In [1]:
import os, json, random
from typing import Dict, List, Optional, Tuple

from datasets import load_dataset
from transformers import AutoTokenizer

/workspace/venvs/qwen/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =========================
# CONFIG
# =========================
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"                  # 用它的 tokenizer 做长度过滤
DATASET_ID = "HuggingFaceH4/ultrachat_200k"   
SPLIT      = "train_sft"                       # UltraChat 常见 split

OUT_DIR = "/root/data/train_data"
SEED = 0

N_CALIB = 1000      # 剪枝/量化校准集。校准集用于forward，帮助压缩算法剪枝/量化时更好地保留重要权重。通常越大越好，但也越慢。
N_SFT   = 10_000    # 恢复训练 SFT 集。用于剪枝后的恢复训练，帮助模型恢复性能。通常越大越好，但也越慢。

# token 长度过滤（用 MODEL_NAME tokenizer）
MIN_PROMPT_TOK = 16
MAX_PROMPT_TOK = 512

MIN_RESP_TOK   = 16
MAX_RESP_TOK   = 512

# 校准集里加一点“更长 prompt”覆盖激活（可设为 0）。少量更长的 prompt 有助于覆盖剪枝/量化后激活分布的变化，提升性能。过多则可能偏离原数据分布，反而不利。
CALIB_LONG_FRAC = 0.10
CALIB_LONG_PROMPT_MAX = 1024

In [3]:
# =========================
# Helpers
# =========================
def set_seed(seed: int) -> None:
    random.seed(seed)

def safe_get_pair_ultrachat(example: Dict) -> Optional[Tuple[str, str]]:
    """
    UltraChat 通常是 example["messages"] = [{"role": "...", "content": "..."}, ...]
    抽取：第一条 user + 其后第一条 assistant
    """
    msgs = example.get("messages")
    if not isinstance(msgs, list) or len(msgs) < 2:
        return None

    u_idx = None
    for i, m in enumerate(msgs):
        if isinstance(m, dict) and m.get("role") == "user":
            u_idx = i
            break
    if u_idx is None:
        return None

    a_idx = None
    for j in range(u_idx + 1, len(msgs)):
        m = msgs[j]
        if isinstance(m, dict) and m.get("role") == "assistant":
            a_idx = j
            break
    if a_idx is None:
        return None

    user = msgs[u_idx].get("content")
    assistant = msgs[a_idx].get("content")
    if not isinstance(user, str) or not isinstance(assistant, str):
        return None

    user = user.strip()
    assistant = assistant.strip()
    if not user or not assistant:
        return None
    return user, assistant

def tok_len(tok: AutoTokenizer, text: str) -> int:
    return len(tok.encode(text, add_special_tokens=False))

def build_record(user: str, assistant: str) -> Dict:
    return {"messages": [{"role": "user", "content": user},
                         {"role": "assistant", "content": assistant}]}

def write_jsonl(path: str, records: List[Dict]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [4]:
# =========================
# Main
# =========================
set_seed(SEED)

print(f"Loading tokenizer: {MODEL_NAME}")
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)

print(f"Loading dataset: {DATASET_ID} split={SPLIT}")
ds = load_dataset(DATASET_ID, split=SPLIT)

idxs = list(range(len(ds)))
random.shuffle(idxs)

target_calib_long  = int(N_CALIB * CALIB_LONG_FRAC)
target_calib_short = N_CALIB - target_calib_long

sft_records: List[Dict] = []
calib_short: List[Dict] = []
calib_long: List[Dict] = []

def accept_short(user: str, assistant: str) -> bool:
    pu = tok_len(tok, user)
    pa = tok_len(tok, assistant)
    return (MIN_PROMPT_TOK <= pu <= MAX_PROMPT_TOK) and (MIN_RESP_TOK <= pa <= MAX_RESP_TOK)

def accept_long(user: str, assistant: str) -> bool:
    pu = tok_len(tok, user)
    pa = tok_len(tok, assistant)
    return (MIN_PROMPT_TOK <= pu <= CALIB_LONG_PROMPT_MAX) and (pu > MAX_PROMPT_TOK) and (MIN_RESP_TOK <= pa <= MAX_RESP_TOK)

for k, i in enumerate(idxs):
    ex = ds[i]
    pair = safe_get_pair_ultrachat(ex)
    if pair is None:
        continue
    user, assistant = pair

    ok_short = accept_short(user, assistant)

    # SFT 数据：用 short 过滤（更稳、更省显存）
    if ok_short and len(sft_records) < N_SFT:
        sft_records.append(build_record(user, assistant))

    # calib short bucket
    if ok_short and len(calib_short) < target_calib_short:
        calib_short.append(build_record(user, assistant))

    # calib long bucket
    if target_calib_long > 0 and len(calib_long) < target_calib_long:
        if accept_long(user, assistant):
            calib_long.append(build_record(user, assistant))

    if (len(sft_records) >= N_SFT and
        len(calib_short) >= target_calib_short and
        len(calib_long) >= target_calib_long):
        break

    if (k + 1) % 50_000 == 0:
        print(f"scanned={k+1} | sft={len(sft_records)}/{N_SFT} | "
              f"calib_short={len(calib_short)}/{target_calib_short} | "
              f"calib_long={len(calib_long)}/{target_calib_long}")

Loading tokenizer: meta-llama/Llama-3.1-8B-Instruct
Loading dataset: HuggingFaceH4/ultrachat_200k split=train_sft


In [5]:
calib_records = calib_short + calib_long
random.shuffle(calib_records)

print("\nFinal counts:")
print(f"  SFT   : {len(sft_records)} (target {N_SFT})")
print(f"  Calib : {len(calib_records)} (target {N_CALIB}) "
      f"[short {len(calib_short)}, long {len(calib_long)}]")

if len(sft_records) < N_SFT:
    raise RuntimeError("Not enough SFT samples. Loosen length filters or change split.")
if len(calib_records) < N_CALIB:
    raise RuntimeError("Not enough calib samples. Loosen filters or set CALIB_LONG_FRAC=0.")

os.makedirs(OUT_DIR, exist_ok=True)
calib_path = os.path.join(OUT_DIR, f"calib_{N_CALIB}.jsonl")
sft_path   = os.path.join(OUT_DIR, f"sft_{N_SFT}.jsonl")

write_jsonl(calib_path, calib_records[:N_CALIB])
write_jsonl(sft_path, sft_records[:N_SFT])

print("\nWrote:")
print(" ", calib_path)
print(" ", sft_path)

print("\nSample calib record:")
print(json.dumps(calib_records[0], ensure_ascii=False, indent=2)[:800])

print("\nSample sft record:")
print(json.dumps(sft_records[0], ensure_ascii=False, indent=2)[:800])


Final counts:
  SFT   : 10000 (target 10000)
  Calib : 1000 (target 1000) [short 900, long 100]

Wrote:
  /root/data/train_data/calib_1000.jsonl
  /root/data/train_data/sft_10000.jsonl

Sample calib record:
{
  "messages": [
    {
      "role": "user",
      "content": "For cycling fans following the Tour de France on social media and mobile screens around the world, we have special treat in store this year. We’re bringing you ever closer to the action, using advanced ways of combining and analysing data through the new algorithms we’ve built into our data analytics platform. These help us create real-time predictions as the race unfolds – a first in pro cycling.\nWhere does all our data come from?\nWe use GPS tracking devices installed underneath each bike’s saddle to track individual riders’ speed, position within the peloton, and distances between riders throughout each stage.\nThird-party data give us environmental information, such as the gradient at which the rider is climbing o